# LeetCode #685: Redundant Connection II

https://leetcode.com/problems/redundant-connection-ii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (Try Removing Each Edge)** | $O(n^2)$ | $O(n)$ |
| **Optimal: Union-Find with Two-Parent Detection ★** | $O(n \cdot \alpha(n))$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force (Try Removing Each Edge)
For each edge, remove it and check if the remaining edges form a valid rooted tree. O(n) per check, O(n) edges.

### Optimal: Union-Find with Two-Parent Detection ★
In a directed graph, the redundant edge either: (1) causes a node to have two parents, or (2) creates a cycle, or (3) both. First, detect if any node has two incoming edges (candidate1 and candidate2). If so, temporarily ignore candidate2 and check for a cycle using Union-Find. If a cycle exists, candidate1 is the answer; otherwise candidate2. If no node has two parents, use standard Union-Find to find the cycle-causing edge.

**Why this is better than Brute Force:** Handles all three cases in a single Union-Find pass plus preprocessing, running in near-linear time.

**Constraints:**
* n == edges.length
* 3 <= n <= 1000
* edges[i].length == 2
* 1 <= ui, vi <= n

## Solutions

### C#

In [ ]:
public class Solution {
    private int[] parent, rank;

    public int[] FindRedundantDirectedConnection(int[][] edges) {
        int n = edges.Length;
        int[] incoming = new int[n + 1];
        int cand1 = -1, cand2 = -1;
        for (int i = 0; i < n; i++) {
            int v = edges[i][1];
            if (incoming[v] != 0) {
                cand1 = incoming[v] - 1;
                cand2 = i;
                break;
            }
            incoming[v] = i + 1;
        }
        parent = new int[n + 1];
        rank = new int[n + 1];
        for (int i = 0; i <= n; i++) parent[i] = i;
        for (int i = 0; i < n; i++) {
            if (i == cand2) continue;
            int u = edges[i][0], v = edges[i][1];
            if (!Union(u, v)) {
                return cand1 >= 0 ? edges[cand1] : edges[i];
            }
        }
        return edges[cand2];
    }

    private int Find(int x) {
        if (parent[x] != x) parent[x] = Find(parent[x]);
        return parent[x];
    }

    private bool Union(int x, int y) {
        int px = Find(x), py = Find(y);
        if (px == py) return false;
        if (rank[px] < rank[py]) parent[px] = py;
        else if (rank[px] > rank[py]) parent[py] = px;
        else { parent[py] = px; rank[px]++; }
        return true;
    }
}

### Python

In [ ]:
class Solution:
    def findRedundantDirectedConnection(self, edges: list[list[int]]) -> list[int]:
        n = len(edges)
        incoming = [0] * (n + 1)
        cand1 = cand2 = -1
        for i, (u, v) in enumerate(edges):
            if incoming[v] != 0:
                cand1 = incoming[v] - 1
                cand2 = i
                break
            incoming[v] = i + 1

        parent = list(range(n + 1))
        rnk = [0] * (n + 1)

        def find(x):
            if parent[x] != x:
                parent[x] = find(parent[x])
            return parent[x]

        def union(x, y):
            px, py = find(x), find(y)
            if px == py:
                return False
            if rnk[px] < rnk[py]:
                px, py = py, px
            parent[py] = px
            if rnk[px] == rnk[py]:
                rnk[px] += 1
            return True

        for i, (u, v) in enumerate(edges):
            if i == cand2:
                continue
            if not union(u, v):
                return edges[cand1] if cand1 >= 0 else edges[i]
        return edges[cand2]

### Go

In [ ]:
func findRedundantDirectedConnection(edges [][]int) []int {
    n := len(edges)
    incoming := make([]int, n+1)
    cand1, cand2 := -1, -1
    for i, e := range edges {
        v := e[1]
        if incoming[v] != 0 {
            cand1 = incoming[v] - 1
            cand2 = i
            break
        }
        incoming[v] = i + 1
    }
    parent := make([]int, n+1)
    rank := make([]int, n+1)
    for i := range parent { parent[i] = i }

    var find func(int) int
    find = func(x int) int {
        if parent[x] != x { parent[x] = find(parent[x]) }
        return parent[x]
    }
    union := func(x, y int) bool {
        px, py := find(x), find(y)
        if px == py { return false }
        if rank[px] < rank[py] { px, py = py, px }
        parent[py] = px
        if rank[px] == rank[py] { rank[px]++ }
        return true
    }

    for i, e := range edges {
        if i == cand2 { continue }
        if !union(e[0], e[1]) {
            if cand1 >= 0 { return edges[cand1] }
            return e
        }
    }
    return edges[cand2]
}

### Rust

In [ ]:
impl Solution {
    pub fn find_redundant_directed_connection(edges: Vec<Vec<i32>>) -> Vec<i32> {
        let n = edges.len();
        let mut incoming = vec![0usize; n + 1];
        let (mut cand1, mut cand2): (i32, i32) = (-1, -1);
        for (i, e) in edges.iter().enumerate() {
            let v = e[1] as usize;
            if incoming[v] != 0 {
                cand1 = (incoming[v] - 1) as i32;
                cand2 = i as i32;
                break;
            }
            incoming[v] = i + 1;
        }
        let mut parent: Vec<usize> = (0..=n).collect();
        let mut rank = vec![0usize; n + 1];
        fn find(p: &mut Vec<usize>, x: usize) -> usize {
            if p[x] != x { p[x] = find(p, p[x]); }
            p[x]
        }
        for (i, e) in edges.iter().enumerate() {
            if i as i32 == cand2 { continue; }
            let (u, v) = (e[0] as usize, e[1] as usize);
            let (pu, pv) = (find(&mut parent, u), find(&mut parent, v));
            if pu == pv {
                return if cand1 >= 0 { edges[cand1 as usize].clone() } else { e.clone() };
            }
            if rank[pu] < rank[pv] { parent[pu] = pv; }
            else if rank[pu] > rank[pv] { parent[pv] = pu; }
            else { parent[pv] = pu; rank[pu] += 1; }
        }
        edges[cand2 as usize].clone()
    }
}

## Example Scenarios

### Scenario 1: Node with two parents
**Input:** `edges = [[1,2],[1,3],[2,3]]`  
Node 3 has two parents (1 and 2). Removing [2,3] leaves a valid tree. **Output:** `[2,3]`

### Scenario 2: Cycle without two-parent node
**Input:** `edges = [[1,2],[2,3],[3,1],[4,1]]`  
No node has two parents, but there is a cycle 1->2->3->1. The last cycle edge [3,1] is the answer. Wait -- node 1 has parents 3 and 4. cand1=[3,1], cand2=[4,1]. Skip [4,1], check cycle: 1->2, 2->3, 3->1 forms cycle with cand1=[3,1]. **Output:** `[3,1]`

### Scenario 3: Two parents and a cycle
**Input:** `edges = [[2,1],[3,1],[1,2]]`  
Node 1 has parents 2 and 3. Edge [1,2] also creates a cycle with [2,1]. The correct answer is [2,1] (the first candidate). **Output:** `[2,1]`

### Scenario 4: Last edge is redundant
**Input:** `edges = [[1,2],[2,3],[3,4],[4,1],[1,5]]`  
Node 1 gets parent from 4 via [4,1]. Removing it breaks the cycle. **Output:** `[4,1]`

### Scenario 5: Simple three-node cycle
**Input:** `edges = [[1,2],[2,1]]` -- minimum valid input has 3 edges  
**Input:** `edges = [[1,2],[2,3],[3,2]]`  
Node 2 has parents 1 and 3. Removing [3,2] keeps a valid tree 1->2->3. **Output:** `[3,2]`

![image](attachment:image.png)